In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.axes import Axes
from typing import cast
try:
    import lightgbm as lgb
    TEM_LGB = True
except ImportError:
    lgb = None
    TEM_LGB = False

# ============================ CONFIGURAÇÃO ============================
BASE = Path(".")                       # rode na pasta do projeto (onde está o painel_modelagem.parquet)
RUN_ID = None                          # coloque o run_id da Parte 2 (ex.: "abc123"); None = rodada cv mais recente
MIN_SETOR = 3                          # mínimo de empresas do setor no treino para usar o IND do setor
MIN_TREINO_BL = 50                     # mínimo de linhas completas no treino do BL (~5 por parâmetro)
UNIDADE_VM = "R$ mil"                  # unidade do valor de mercado no painel (confira na Parte 1)
BL_COM_IND = True                      # inclui no BL o múltiplo médio do setor como variável (como em Bhojraj & Lee)
VARS_BL = ["roe", "roa", "margem_liq", "margem_ebit", "giro_ativos",
           "divida_pl", "cresc_vendas", "cresc_ativos", "ln_ativos", "beta"]
MODELOS = ["GBM", "IND", "BL", "BL_log"] + (["GBM10"] if TEM_LGB else [])   # GBM10 = GBM só com as variáveis do BL
PARAMS_GBM10 = dict(objective="regression", learning_rate=0.05, num_leaves=15, min_data_in_leaf=20,
                    bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, seed=0, num_threads=1, verbose=-1)
ALVOS_EV = ["lnv2a", "lnv2s"]

# ============================ 1) ESCOLHA DA RODADA E LEITURA ============================
if RUN_ID:
    PASTA = BASE / "resultados" / f"run_{RUN_ID}"
else:
    runs = [p.parent for p in BASE.glob("resultados/run_*/previsoes_oos.parquet")
            if json.loads((p.parent / "config.json").read_text())["modo"] == "cv"]
    if not runs:
        raise FileNotFoundError("Nenhuma rodada em modo cv. Rode a Parte 2 com --modo cv --n-meses 0.")
    PASTA = max(runs, key=lambda p: (p / "previsoes_oos.parquet").stat().st_mtime)   # data do arquivo da Parte 2
    print(f"AVISO: RUN_ID não definido, usando a rodada cv mais recente ({PASTA.name}).")

config = json.loads((PASTA / "config.json").read_text())
if config["modo"] != "cv":
    raise ValueError("A rodada escolhida não está em modo cv (não tem blocos).")
if config.get("n_meses") not in (0, None):
    print(f"AVISO: rodada reduzida (n_meses = {config['n_meses']}). Para o resultado final use --n-meses 0.")

SAIDA = PASTA / "etapa3"
SAIDA.mkdir(exist_ok=True)
print("Usando:", PASTA.name)

prev = pd.read_parquet(PASTA / "previsoes_oos.parquet")
painel = pd.read_parquet(BASE / "painel_modelagem.parquet")
prev["data"] = pd.to_datetime(prev["data"])
painel["data"] = pd.to_datetime(painel["data"])
prev = prev[prev["bloco"] >= 0]

# setor é opcional: sem a coluna, o IND vira a média harmônica de todas as empresas do mês (um "setor" único)
if "setor" not in painel.columns:
    print("AVISO: painel sem coluna 'setor'. IND = média harmônica do mercado inteiro e não há análise por setor.")
    painel["setor"] = "todos"
# variável do BL faltando gera aviso
faltando = [c for c in VARS_BL if c not in painel.columns]
if faltando:
    print("AVISO: variáveis do BL ausentes no painel (descartadas):", faltando)
vars_bl = [c for c in VARS_BL if c in painel.columns]
print("Variáveis do BL:", vars_bl)

# variáveis originais (sem corte) para os quintis
feat = painel[["ticker", "data", "setor"] + vars_bl].copy()
for c in ["ln_ativos", "roe", "pl"]:
    if c in painel.columns:
        feat[f"orig_{c}"] = painel[c]

# limita outliers das variáveis do BL (corte 1%-99% dentro de cada mês)
feat[vars_bl] = feat.groupby("data")[vars_bl].transform(lambda s: s.clip(s.quantile(.01), s.quantile(.99)))

# junta as previsões do GBM com as variáveis do painel (validate: erro se o painel tiver empresa-mês duplicada)
df = prev[["ticker", "data", "alvo", "bloco", "log_real", "pred_log",
           "valor_mercado", "ev", "denominador"]].merge(feat, on=["ticker", "data"], how="left",
                                                        validate="many_to_one")
df["setor"] = df["setor"].astype(object)
df["orig_ln_vm"] = np.log(df["valor_mercado"].where(df["valor_mercado"] > 0))   # tamanho por valor de mercado
if "orig_pl" in df:
    df["orig_mb"] = (df["valor_mercado"] / df["orig_pl"]).where(df["orig_pl"] > 0)   # market-to-book
df = df.reset_index(drop=True)


# ============================ 2) BENCHMARKS (IND, BL, BL_log e GBM10) ============================
def hmean(m):                                   # média harmônica
    return len(m) / np.sum(1 / m)


def benchmarks_mes(g):
    """Para cada bloco k do mês: treina nos outros blocos e prevê o bloco k (mesma partição do GBM)."""
    g = g.reset_index(drop=True)
    mult = np.exp(g["log_real"].to_numpy(float))        # múltiplo real (m2b, v2a ou v2s)
    completo = g[vars_bl].notna().all(axis=1).to_numpy()
    y_log = g["log_real"].to_numpy(float)
    for c in ["mult_IND", "mult_BL", "mult_BL_log", "mult_GBM10"]:
        g[c] = np.nan
    blocos = sorted(g["bloco"].unique())
    for i, k in enumerate(blocos):
        te = np.asarray(g["bloco"] == k, dtype=bool)
        tr = ~te
        if tr.sum() < 5:
            continue
        # IND: média harmônica do setor no treino (setor pequeno ou sem setor -> média do mês)
        h_setor = pd.Series(mult[tr]).groupby(g.loc[tr, "setor"].to_numpy()).agg(
            lambda m: hmean(m) if len(m) >= MIN_SETOR else np.nan)
        ind = g["setor"].map(h_setor).fillna(hmean(mult[tr])).to_numpy()   # múltiplo do setor (só com o treino)
        g.loc[te, "mult_IND"] = ind[te]
        # BL: regressão OLS nas variáveis fundamentalistas (só linhas completas)
        X = g[vars_bl].to_numpy(float)
        if BL_COM_IND:
            X = np.column_stack([X, ind])                   # + múltiplo médio do setor
        ok_tr, ok_te = tr & completo, te & completo
        if ok_tr.sum() >= MIN_TREINO_BL and ok_te.any():
            A = np.column_stack([np.ones(ok_tr.sum()), X[ok_tr]])
            B = np.column_stack([np.ones(ok_te.sum()), X[ok_te]])
            b = np.linalg.lstsq(A, mult[ok_tr], rcond=None)[0]                          # alvo = múltiplo
            g.loc[ok_te, "mult_BL"] = B @ b
            b = np.linalg.lstsq(A, g.loc[ok_tr, "log_real"].to_numpy(float), rcond=None)[0]   # alvo = log
            g.loc[ok_te, "mult_BL_log"] = np.exp(B @ b)
        # GBM10: LightGBM só com as variáveis do BL (alvo em log); o bloco seguinte ao teste é a validação
        if TEM_LGB:
            va = np.asarray(g["bloco"] == blocos[(i + 1) % len(blocos)], dtype=bool) & tr
            fit = tr & ~va
            if fit.sum() >= MIN_TREINO_BL and va.sum() >= 5:
                Xg = g[vars_bl].to_numpy(float)
                d_fit = lgb.Dataset(Xg[fit], y_log[fit])
                d_va = lgb.Dataset(Xg[va], y_log[va], reference=d_fit)
                m = lgb.train(PARAMS_GBM10, d_fit, num_boost_round=1000, valid_sets=[d_va],
                              callbacks=[lgb.early_stopping(50, verbose=False)])
                g.loc[te, "mult_GBM10"] = np.exp(m.predict(Xg[te], num_iteration=m.best_iteration))
    return g


df = pd.concat([benchmarks_mes(g) for _, g in df.groupby(["alvo", "data"])], ignore_index=True)
df["mult_GBM"] = np.exp(df["pred_log"])

# média do treino (outros blocos do mesmo alvo-mês), usada no R² fora da amostra
for c in ["valor_mercado", "log_real"]:
    tot = df.groupby(["alvo", "data"])[c]
    blo = df.groupby(["alvo", "data", "bloco"])[c]
    df[f"{c}_treino"] = (tot.transform("sum") - blo.transform("sum")) / (tot.transform("count") - blo.transform("count"))

# quintis dentro de cada mês (1 = menor), com as variáveis originais
QUINTIS_TODOS = {"ln_vm": "tamanho (ln valor de mercado)", "ln_ativos": "tamanho (ln ativos)",
                 "mb": "market-to-book", "roe": "ROE"}
QUINTIS = {c: nome for c, nome in QUINTIS_TODOS.items() if f"orig_{c}" in df}
for c, nome in QUINTIS_TODOS.items():
    if c not in QUINTIS:
        print(f"AVISO: quintil de {nome} NÃO gerado (coluna ausente no painel).")
for c in QUINTIS:
    df[f"q_{c}"] = df.groupby(["alvo", "data"])[f"orig_{c}"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 5, labels=False) + 1 if s.notna().sum() >= 5 else np.nan)

# ============================ 3) CONVERSÃO PARA VALOR DE MERCADO E ERRO ============================
# lnm2b: VM_prev = múltiplo * PL | lnv2a/lnv2s: VM_prev = múltiplo * denominador - (EV - VM)
dl = np.where(df["alvo"].isin(ALVOS_EV), df["ev"] - df["valor_mercado"], 0.0)
cols = ["ticker", "data", "alvo", "bloco", "setor", "valor_mercado", "log_real",
        "valor_mercado_treino", "log_real_treino"] + [f"q_{c}" for c in QUINTIS]
linhas = []
for m in MODELOS:
    r = df[cols].copy()
    r["modelo"] = m
    r["mult_prev"] = df[f"mult_{m}"]
    r["vm_prev"] = r["mult_prev"] * df["denominador"] - dl
    r["eps"] = r["vm_prev"] / r["valor_mercado"] - 1                                  # erro de avaliação
    r["erro_log"] = np.log(r["mult_prev"].where(r["mult_prev"] > 0)) - r["log_real"]  # erro no log do múltiplo
    linhas.append(r)
res = pd.concat(linhas, ignore_index=True).dropna(subset=["mult_prev"])

# amostra comum: só observações que TODOS os modelos conseguiram prever
n_mod = res.groupby(["ticker", "data", "alvo"])["modelo"].transform("nunique")
comum = res[n_mod == len(MODELOS)].copy()

# base observação a observação (auditoria e Parte 4)
res.to_parquet(SAIDA / "previsoes_etapa3.parquet", index=False)


# ============================ 4) MÉTRICAS ============================
def metricas(d):
    err = d["vm_prev"] - d["valor_mercado"]
    ok = d["erro_log"].notna()
    el, lr, lt = d.loc[ok, "erro_log"], d.loc[ok, "log_real"], d.loc[ok, "log_real_treino"]
    return pd.Series({
        "n": len(d),
        "med_eps": d["eps"].median(),                       # viés (perto de 0 = bom)
        "med_abs_eps": d["eps"].abs().median(),             # acurácia (menor = melhor)
        "p5_eps": d["eps"].quantile(.05),
        "p95_eps": d["eps"].quantile(.95),
        "frac_vm_neg": (d["vm_prev"] <= 0).mean(),          # fração de valor previsto <= 0
        "rmse_vm": np.sqrt((err ** 2).mean()),              # em UNIDADE_VM
        "r2_vm": 1 - (err ** 2).sum() / ((d["valor_mercado"] - d["valor_mercado_treino"]) ** 2).sum(),
        "corr": d["vm_prev"].corr(d["valor_mercado"]),
        "rmse_log": np.sqrt((el ** 2).mean()),              # no log do múltiplo
        "r2_log": 1 - (el ** 2).sum() / ((lr - lt) ** 2).sum(),
    })


def tabela(d, chaves):
    t = pd.DataFrame({k: metricas(g) for k, g in d.groupby(chaves)}).T.rename_axis(index=chaves)
    t["n"] = t["n"].astype(int)
    return t.reindex(MODELOS, level="modelo")             # GBM, IND, BL, BL_log nessa ordem


def mediana_abs(d, chaves):
    """Mediana de |ε| por grupo, uma coluna por modelo."""
    return d.groupby(chaves + ["modelo"])["eps"].apply(lambda s: s.abs().median()).unstack("modelo")[MODELOS]


tab_todas = tabela(res, ["alvo", "modelo"])
tab_comum = tabela(comum, ["alvo", "modelo"])
tab_todas.to_csv(SAIDA / "tabela_principal_todas.csv")
tab_comum.to_csv(SAIDA / "tabela_principal_comum.csv")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.3f}".format)
print(f"\nRMSE em valor de mercado em {UNIDADE_VM}. R² calculado com a média do treino.")
print("\n=== TODAS AS PREVISÕES DE CADA MODELO (n diferente por modelo) ===\n", tab_todas)
print("\n=== AMOSTRA COMUM (comparação justa) ===\n", tab_comum)

# cobertura: quanto o BL conseguiu prever e quantos meses-bloco ficaram sem BL
bl_bloco = df.groupby(["alvo", "data", "bloco"])["mult_BL"].apply(lambda s: s.notna().any())
cob = pd.DataFrame({
    "pct_obs_com_BL": df.groupby("alvo")["mult_BL"].apply(lambda s: s.notna().mean()),
    "blocos_sem_BL": (~bl_bloco).groupby("alvo").sum(),
    "blocos_total": bl_bloco.groupby("alvo").size(),
})
cob.to_csv(SAIDA / "cobertura_bl.csv")
print(f"\n=== COBERTURA DO BL (MIN_TREINO_BL = {MIN_TREINO_BL}) ===\n", cob)

# conferência: o med(|ε|) do GBM aqui deve bater com o da Parte 2
if "erro_pct_vm" in prev:
    conf = pd.DataFrame({"parte2": prev.groupby("alvo")["erro_pct_vm"].apply(lambda s: s.abs().median()),
                         "parte3": tab_todas.xs("GBM", level="modelo")["med_abs_eps"]})
    print("\n=== CONFERÊNCIA COM A PARTE 2: med(|ε|) do GBM ===\n", conf)

# reconciliação (Tabela 4): BL -> (a) log no alvo -> (b) OLS vira GBM -> (c) todas as features
passos = {"BL": "BL (OLS)", "BL_log": "(a) + log no alvo"}
if TEM_LGB:
    passos["GBM10"] = "(b) OLS -> GBM (só vars do BL)"
passos["GBM"] = "(c) + todas as features (GBM Parte 2)"
rec = tab_comum.reset_index()
rec = rec[rec["modelo"].isin(passos)]
rec["passo"] = rec["modelo"].map(passos)
rec["ordem"] = rec["modelo"].map({m: i for i, m in enumerate(passos)})
rec = rec.sort_values(["alvo", "ordem"]).set_index(["alvo", "passo"])[
    ["n", "med_abs_eps", "rmse_vm", "r2_vm", "rmse_log", "r2_log"]]
rec.to_csv(SAIDA / "tabela_reconciliacao.csv")
print("\n=== RECONCILIAÇÃO (amostra comum) ===\n", rec)

# ============================ 5) GRÁFICOS: MÊS A MÊS, QUINTIS E SETOR ============================
alvos = sorted(comum["alvo"].unique())

# erro mês a mês
mensal = mediana_abs(comum, ["alvo", "data"])
mensal.to_csv(SAIDA / "erro_por_mes.csv")
fig, axs = plt.subplots(1, len(alvos), figsize=(6 * len(alvos), 4), squeeze=False, sharey=True)
for ax, a in zip(axs[0], alvos):
    mensal.loc[a].plot(ax=ax, linewidth=1, title=a)
    ax.set_ylabel("mediana de |ε|")
plt.tight_layout()
plt.savefig(SAIDA / "erro_por_mes.png", dpi=150)
plt.close()

# erro por quintil (uma linha da figura por alvo, uma coluna por variável)
fig, axs = plt.subplots(len(alvos), len(QUINTIS), figsize=(4.5 * len(QUINTIS), 3.5 * len(alvos)), squeeze=False)
for j, (c, nome) in enumerate(QUINTIS.items()):
    q = mediana_abs(comum.dropna(subset=[f"q_{c}"]), ["alvo", f"q_{c}"])
    q.to_csv(SAIDA / f"erro_por_quintil_{c}.csv")
    print(f"\n=== mediana de |ε| por quintil de {nome} (1 = menor) ===\n", q)
    for i, a in enumerate(alvos):
        ax = cast(Axes, axs[i, j])

        q.loc[a].plot(
            ax=ax,
            marker="o",
            legend=(i == 0 and j == 0),
            title=f"{a} - {nome}"
        )

        ax.set_xlabel("quintil")
plt.tight_layout()
plt.savefig(SAIDA / "erro_por_quintil.png", dpi=150)
plt.close()

# erro por setor (só se o painel tiver setor)
st = mediana_abs(comum, ["alvo", "setor"])
if comum["setor"].nunique() > 1:
    st.to_csv(SAIDA / "erro_por_setor.csv")
    print("\n=== mediana de |ε| por setor ===\n", st)
    fig, axs = plt.subplots(1, len(alvos), figsize=(6 * len(alvos), 0.4 * st.loc[alvos[0]].shape[0] + 2),
                            squeeze=False, sharey=True)
    for ax, a in zip(axs[0], alvos):
        st.loc[a].plot.barh(ax=ax, width=0.8, legend=(a == alvos[-1]), title=a)
        ax.set_xlabel("mediana de |ε|")
    plt.tight_layout()
    plt.savefig(SAIDA / "erro_por_setor.png", dpi=150)
    plt.close()

print("\nArquivos salvos em:", SAIDA)